# Flight price prediction
Compare four models using the same flights. Choose the model with the lowest validation RMSE, check it once on the test set, and save **all four fitted models and their metadata** in `training/`. The `models/` folder is reserved for registered versions. Put `train.csv` beside this notebook and run the cells in order.

In [1]:
# If needed, uncomment and run once in your notebook environment, then restart the kernel.
# %pip install pandas scikit-learn xgboost lightgbm catboost
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

### Prepare the data
The original notebook trained a Random Forest using airline, cities, stops, journey date, clock times and duration. It also tried tuning the forest, but saved the untuned model. Here all four models get the same information.

In [2]:
data = pd.read_csv('train.csv')
data = data.loc[:, ~data.columns.str.startswith('Unnamed')].drop_duplicates().copy()
columns = ['Airline', 'Source', 'Destination', 'Total_Stops', 'Date_of_Journey',
           'Dep_Time', 'Arrival_Time', 'Duration', 'Price']
data = data[columns].dropna().copy()

# Convert durations such as "2h 45m" into minutes.
hours = data['Duration'].str.extract(r'(\d+)h')[0].fillna(0).astype(int)
minutes = data['Duration'].str.extract(r'(\d+)m')[0].fillna(0).astype(int)
data['minutes'] = 60 * hours + minutes
data = data[data['minutes'] >= 30].copy()  # removes a five-minute, two-stop itinerary

date = pd.to_datetime(data['Date_of_Journey'], format='%d/%m/%Y')
X = data[['Airline', 'Source', 'Destination']].copy()
X['stops'] = data['Total_Stops'].replace('non-stop', '0').str.split().str[0].astype(int)
X['month'] = date.dt.month
X['day'] = date.dt.day
departure = pd.to_datetime(data['Dep_Time'], format='%H:%M')
arrival = pd.to_datetime(data['Arrival_Time'].str.split().str[0], format='%H:%M')
X['departure_hour'] = departure.dt.hour
X['departure_minute'] = departure.dt.minute
X['arrival_hour'] = arrival.dt.hour
X['arrival_minute'] = arrival.dt.minute
X['minutes'] = data['minutes']
y = data['Price']
print(len(X), 'flights')
display(X.head())

7375 flights


C:\Users\keith\AppData\Local\Temp\ipykernel_120008\1401150416.py:8: Pandas4Warning: 'int' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  hours = data['Duration'].str.extract(r'(\d+)h')[0].fillna(0).astype(int)
C:\Users\keith\AppData\Local\Temp\ipykernel_120008\1401150416.py:9: Pandas4Warning: 'int' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  minutes = data['Duration'].str.extract(r'(\d+)m')[0].fillna(0).astype(int)


,Airline,Source,Destination,stops,month,day,departure_hour,departure_minute,arrival_hour,arrival_minute,minutes
0,Air India,Delhi,Cochin,1,6,3,3,50,19,15,925
1,Air India,Delhi,Cochin,1,5,18,23,0,7,40,520
2,IndiGo,Delhi,Cochin,1,6,3,8,35,22,30,835
3,Jet Airways,Kolkata,Banglore,1,5,21,21,10,16,20,1150
4,IndiGo,Banglore,Delhi,0,6,12,6,5,8,50,165


### Split the flights
Split by journey date so that repeated copies of a flight stay together. About 60% of dates go to training, 20% to validation and 20% to testing. This measures performance on other dates in the same historical period.

In [3]:
# train_dates, other_dates = train_test_split(sorted(date.unique()), test_size=0.4, random_state=42)
# valid_dates, test_dates = train_test_split(other_dates, test_size=0.5, random_state=42)
# train = date.isin(train_dates)
# valid = date.isin(valid_dates)
# test = date.isin(test_dates)
# print('Training:', train.sum(), 'Validation:', valid.sum(), 'Test:', test.sum())

In [4]:
train_dates, other_dates = train_test_split(
    sorted(date.unique()), test_size=0.4, random_state=42
)

valid_dates, test_dates = train_test_split(
    other_dates, test_size=0.5, random_state=42
)

# Reserve some training dates for calibrating prediction intervals.
train_dates, calibration_dates = train_test_split(
    train_dates, test_size=0.25, random_state=42
)

train = date.isin(train_dates)
valid = date.isin(valid_dates)
calibration = date.isin(calibration_dates)
test = date.isin(test_dates)

print(
    'Training:', train.sum(),
    'Validation:', valid.sum(),
    'Calibration:', calibration.sum(),
    'Test:', test.sum()
)

Training: 3831 Validation: 1499 Calibration: 712 Test: 1333


### Train and compare the models
One-hot encode airline, source, and destination for Random Forest, XGBoost, and LightGBM. CatBoost encodes those same categories internally. Departure and arrival hours/minutes remain numeric. Each model learns preprocessing from training rows only. Higher R² and lower MAE or RMSE are better. Errors are in rupees.

In [5]:
text_columns = ['Airline', 'Source', 'Destination']
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, n_jobs=1, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=200, n_jobs=1, verbosity=-1, random_state=42),
    'CatBoost': CatBoostRegressor(iterations=200, thread_count=1, random_seed=42,
                                 verbose=False, allow_writing_files=False,
                                 cat_features=text_columns)
}
results = []
fitted = {}
for name, estimator in models.items():
    if name == 'CatBoost':
        # CatBoost handles categorical features itself and accepts the same DataFrame.
        model = estimator
    else:
        encoder = ColumnTransformer([('text', OneHotEncoder(handle_unknown='ignore',
                                    sparse_output=False), text_columns)], remainder='passthrough')
        model = make_pipeline(encoder, estimator)

    model.fit(X[train], y[train].to_numpy())
    prediction = model.predict(X[valid])
    results.append({'Model': name,
                    'Train R²': model.score(X[train], y[train]),
                    'Validation R²': r2_score(y[valid], prediction),
                    'MAE': mean_absolute_error(y[valid], prediction),
                    'RMSE': mean_squared_error(y[valid], prediction) ** 0.5})
    fitted[name] = model

scores = pd.DataFrame(results).set_index('Model').sort_values('RMSE')
display(scores.round(3))

,Train R²,Validation R²,MAE,RMSE
Model,,,,
LightGBM,0.919,0.782,1285.459,1824.484
Random Forest,0.956,0.755,1301.382,1936.022
CatBoost,0.901,0.752,1371.872,1947.355
XGBoost,0.969,0.738,1285.066,2001.232


In [6]:
fitted['Random Forest'].feature_names_in_

array(['Airline', 'Source', 'Destination', 'stops', 'month', 'day',
       'departure_hour', 'departure_minute', 'arrival_hour',
       'arrival_minute', 'minutes'], dtype=object)

In [7]:
help(fitted['Random Forest'])

Help on Pipeline in module sklearn.pipeline object:

class Pipeline(sklearn.callback._callback_support.CallbackSupportMixin, sklearn.utils.metaestimators._BaseComposition)
 |  Pipeline(steps, *, transform_input=None, memory=None, verbose=False)
 |
 |  A sequence of data transformers with an optional final predictor.
 |
 |  `Pipeline` allows you to sequentially apply a list of transformers to
 |  preprocess the data and, if desired, conclude the sequence with a final
 |  :term:`predictor` for predictive modeling.
 |
 |  Intermediate steps of the pipeline must be transformers, that is, they
 |  must implement `fit` and `transform` methods.
 |  The final :term:`estimator` only needs to implement `fit`.
 |  The transformers in the pipeline can be cached using ``memory`` argument.
 |
 |  The purpose of the pipeline is to assemble several steps that can be
 |  cross-validated together while setting different parameters. For this, it
 |  enables setting parameters of the various steps using t

In [8]:
feature_names = fitted['Random Forest'][:-1].get_feature_names_out()
feature_names

array(['text__Airline_Air Asia', 'text__Airline_Air India',
       'text__Airline_GoAir', 'text__Airline_IndiGo',
       'text__Airline_Jet Airways', 'text__Airline_Jet Airways Business',
       'text__Airline_Multiple carriers',
       'text__Airline_Multiple carriers Premium economy',
       'text__Airline_SpiceJet', 'text__Airline_Vistara',
       'text__Airline_Vistara Premium economy', 'text__Source_Banglore',
       'text__Source_Chennai', 'text__Source_Delhi',
       'text__Source_Kolkata', 'text__Source_Mumbai',
       'text__Destination_Banglore', 'text__Destination_Cochin',
       'text__Destination_Delhi', 'text__Destination_Hyderabad',
       'text__Destination_Kolkata', 'text__Destination_New Delhi',
       'remainder__stops', 'remainder__month', 'remainder__day',
       'remainder__departure_hour', 'remainder__departure_minute',
       'remainder__arrival_hour', 'remainder__arrival_minute',
       'remainder__minutes'], dtype=object)

### Test the winner and save all four models
Choose the winner using validation RMSE. Refit each model on the training and validation rows, then save its fitted model and matching metadata to `training/`. The registered versions in `models/` are not changed by this notebook. Evaluate only the selected winner on the held-out test dates.

In [9]:
from pathlib import Path
import json
from datetime import datetime, timezone

training_dir = Path('training')
training_dir.mkdir(exist_ok=True)

model_files = {
    'Random Forest': 'random_forest.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl',
    'CatBoost': 'catboost.pkl',
}

# Refit and save each model along with its validation results.
for name, trained_pipeline in fitted.items():
    trained_pipeline.fit(X[train | valid], y[train | valid].to_numpy())

    path = training_dir / model_files[name]
    with open(path, 'wb') as file:
        pickle.dump(trained_pipeline, file)

    metadata = {
        'algorithm': name,
        'features': X.columns.tolist(),
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'validation_r2': float(scores.loc[name, 'Validation R²']),
        'validation_mae': float(scores.loc[name, 'MAE']),
        'validation_rmse': float(scores.loc[name, 'RMSE'])
    }

    metadata_path = training_dir / f'{path.stem}_metadata.json'
    with open(metadata_path, 'w') as file:
        json.dump(metadata, file, indent=4)

    print('Saved:', path, 'and', metadata_path)

# Evaluate the validation-selected winner once on the test set.
winner = scores.index[0]
model = fitted[winner]
prediction = model.predict(X[test])
print('Selected model:', winner)
print('Test R²:', round(r2_score(y[test], prediction), 3))
print('Test MAE:', round(mean_absolute_error(y[test], prediction), 2))
print('Test RMSE:', round(mean_squared_error(y[test], prediction) ** 0.5, 2))

Saved: training\random_forest.pkl and training\random_forest_metadata.json
Saved: training\xgboost.pkl and training\xgboost_metadata.json
Saved: training\lightgbm.pkl and training\lightgbm_metadata.json
Saved: training\catboost.pkl and training\catboost_metadata.json
Selected model: LightGBM
Test R²: 0.711
Test MAE: 1284.61
Test RMSE: 2058.62


In [13]:
calibration_errors = {}

for name, model in fitted.items():
    predictions = model.predict(X[calibration])
    errors = abs(y[calibration].to_numpy() - predictions)
    calibration_errors[name] = errors

    print(name, '| Calibration flights:', len(errors))

Random Forest | Calibration flights: 712
XGBoost | Calibration flights: 712
LightGBM | Calibration flights: 712
CatBoost | Calibration flights: 712


In [14]:
import numpy as np
from math import ceil

coverage = 0.90
interval_margins = {}

for name, errors in calibration_errors.items():
    rank = ceil((len(errors) + 1) * coverage)
    margin = float(np.sort(errors)[rank - 1])
    interval_margins[name] = margin

    print(f'{name} | Interval margin: ±₹{margin:,.2f}')

Random Forest | Interval margin: ±₹4,412.01
XGBoost | Interval margin: ±₹4,724.06
LightGBM | Interval margin: ±₹5,104.24
CatBoost | Interval margin: ±₹5,172.55


In [15]:
model = fitted[winner]
predictions = model.predict(X[test])
margin = interval_margins[winner]

lower = np.maximum(0, predictions - margin)
upper = predictions + margin
actual = y[test].to_numpy()

test_coverage = np.mean((actual >= lower) & (actual <= upper))

print('Selected model:', winner)
print(f'Test interval coverage: {test_coverage:.1%}')
print(f'Average interval width: ₹{np.mean(upper - lower):,.2f}')

Selected model: LightGBM
Test interval coverage: 98.3%
Average interval width: ₹9,979.56


In [16]:
for name, margin in interval_margins.items():
    model_path = training_dir / model_files[name]
    metadata_path = training_dir / f'{model_path.stem}_metadata.json'

    with open(metadata_path, 'r') as file:
        metadata = json.load(file)

    metadata['prediction_interval'] = {
        'coverage': coverage,
        'margin': margin
    }

    with open(metadata_path, 'w') as file:
        json.dump(metadata, file, indent=4)

    print('Updated:', metadata_path)

Updated: training\random_forest_metadata.json
Updated: training\xgboost_metadata.json
Updated: training\lightgbm_metadata.json
Updated: training\catboost_metadata.json


The results above are calculated using the current feature representation. Compare validation performance before selecting a model. The scores from the previous notebook version are no longer applicable after changing how departure and arrival times are encoded.

### Example prediction using the Flask form fields
The model takes one row with the columns shown above. This example converts existing form fields into the same numerical hour and minute features used during training. Using both dates gives the correct duration for an overnight flight.

In [17]:
model_path = training_dir / model_files[winner]
metadata_path = training_dir / f'{model_path.stem}_metadata.json'

with open(model_path, 'rb') as file:
    saved_model = pickle.load(file)

with open(metadata_path, 'r') as file:
    metadata = json.load(file)

prediction = float(saved_model.predict(flight)[0])
margin = metadata['prediction_interval']['margin']
coverage = metadata['prediction_interval']['coverage']

lower = max(0, prediction - margin)
upper = prediction + margin

print(f'Predicted fare: ₹{prediction:,.2f}')
print(f'{coverage:.0%} prediction interval: ₹{lower:,.2f}–₹{upper:,.2f}')

Predicted fare: ₹8,749.93
90% prediction interval: ₹3,645.68–₹13,854.17
